In [0]:
from pyspark.sql.functions import col, coalesce, when, to_date

# Load both source tables
fotmob_df = spark.read.table("workspace.fotmob.player_overview_processed")
sofascore_df = spark.read.table("workspace.fotmob.sofascore_player_overview_processed")

# Normalize birthdate to date type for both dataframes
# Fotmob has it as string (timestamp), Sofascore has it as date already
fotmob_df = fotmob_df.withColumn(
    "birthdate",
    to_date(col("birthdate"))
)

# Map Sofascore primary positions and filter out goalkeepers
# M --> Midfielder, F --> Forward, D --> Defender, G --> excluded
sofascore_df = sofascore_df.withColumn(
    "primary_position",
    when(col("primary_position") == "M", "Midfielder")
    .when(col("primary_position") == "F", "Forward")
    .when(col("primary_position") == "D", "Defender")
    .otherwise(col("primary_position"))  # Keep other values as-is
).filter(col("primary_position") != "G")  # Exclude goalkeepers

print(f"Fotmob overviews: {fotmob_df.count()} rows, {len(fotmob_df.columns)} columns")
print(f"Sofascore overviews: {sofascore_df.count()} rows, {len(sofascore_df.columns)} columns")

# Identify overlapping columns (excluding join key)
join_key = 'player_id'
fotmob_cols = set(fotmob_df.columns)
sofascore_cols = set(sofascore_df.columns)

overlapping_cols = (fotmob_cols & sofascore_cols) - {join_key}
fotmob_only = fotmob_cols - sofascore_cols - {join_key}
sofascore_only = sofascore_cols - fotmob_cols - {join_key}

print(f"\nJoin key: {join_key}")
print(f"\nOverlapping columns ({len(overlapping_cols)}): {sorted(overlapping_cols)}")
print(f"\nFotmob-only columns ({len(fotmob_only)}): {sorted(fotmob_only)}")
print(f"\nSofascore-only columns ({len(sofascore_only)}): {sorted(sofascore_only)}")

Fotmob overviews: 1488 rows, 9 columns
Sofascore overviews: 482 rows, 9 columns

Join key: player_id

Overlapping columns (8): ['birthdate', 'club', 'club_id', 'country', 'player_name', 'preferred_foot', 'primary_position', 'secondary_positions']

Fotmob-only columns (0): []

Sofascore-only columns (0): []


In [0]:
# Rename overlapping columns in sofascore with _sofascore suffix
for col_name in overlapping_cols:
    sofascore_df = sofascore_df.withColumnRenamed(col_name, f"{col_name}_sofascore")

# Perform full outer join on player_id
gold_df = fotmob_df.alias("fotmob").join(
    sofascore_df.alias("sofascore"),
    on=join_key,
    how="full_outer"
)

print(f"After join: {gold_df.count()} rows")

# For overlapping columns, coalesce to prefer fotmob values (fotmob first, then sofascore)
for col_name in overlapping_cols:
    gold_df = gold_df.withColumn(
        col_name,
        coalesce(col(f"fotmob.{col_name}"), col(f"{col_name}_sofascore"))
    ).drop(f"{col_name}_sofascore")

print(f"\nGold table will have {len(gold_df.columns)} columns")
print("\nSample of combined data:")
display(gold_df.limit(5))

After join: 1970 rows

Gold table will have 9 columns

Sample of combined data:


player_id,player_name,birthdate,club_id,club,primary_position,secondary_positions,preferred_foot,country
180455,Kosovare Asllani,1989-07-29,1075419,London City Lionesses,Attacking Midfielder,"List(Central Midfielder, Striker)",right,Sweden
271109,Alexandra Popp,1991-04-06,394121,VfL Wolfsburg,Striker,"List(Central Midfielder, Attacking Midfielder, Right Winger)",left,Germany
440499,Tuva Hansen,1997-08-04,231497,West Ham United,Center Back,List(Right Back),right,Norway
646111,Noëlle Maritz,1995-12-23,231494,Aston Villa,Center Back,"List(Left Wing-Back, Right Midfielder, Left Winger)",right,Switzerland
734720,Ewa Pajor,1996-12-03,401657,Barcelona,Striker,List(Right Winger),right,Poland


In [0]:
from pyspark.sql.functions import lit, current_timestamp

# Add metadata columns to track data source
gold_df = gold_df.withColumn(
    "data_source",
    when(col("fotmob.player_id").isNotNull() & col("sofascore.player_id").isNotNull(), lit("both"))
    .when(col("fotmob.player_id").isNotNull(), lit("fotmob_only"))
    .otherwise(lit("sofascore_only"))
).withColumn(
    "last_updated",
    current_timestamp()
)

# Show source distribution
print("Data source distribution:")
gold_df.groupBy("data_source").count().orderBy("data_source").show()

# Clean up the aliased join column (remove fotmob. and sofascore. prefixes)
if f"fotmob.{join_key}" in gold_df.columns:
    gold_df = gold_df.withColumn(
        join_key,
        coalesce(col(f"fotmob.{join_key}"), col(f"sofascore.{join_key}"))
    ).drop(f"fotmob.{join_key}").drop(f"sofascore.{join_key}")

Data source distribution:
+--------------+-----+
|   data_source|count|
+--------------+-----+
|   fotmob_only| 1488|
|sofascore_only|  482|
+--------------+-----+



In [0]:
from delta.tables import DeltaTable

# Save to gold table
gold_table_name = "workspace.fotmob.player_overview_gold"

# Count rows
row_count = gold_df.count()
print(f"Writing {row_count} rows to gold table...")

# If table doesn't exist, create it
if not spark.catalog.tableExists(gold_table_name):
    gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
    print(f"\n✅ Created gold table {gold_table_name} with {row_count} rows")
else:
    # Table exists - use MERGE to upsert
    delta_table = DeltaTable.forName(spark, gold_table_name)
    delta_table.alias("target").merge(
        gold_df.alias("source"),
        "target.player_id = source.player_id"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print(f"\n✅ Merged {row_count} records into {gold_table_name}")

# Show final stats
final_count = spark.read.table(gold_table_name).count()
final_cols = len(spark.read.table(gold_table_name).columns)

print(f"\nFinal gold table stats:")
print(f"  - Total rows: {final_count}")
print(f"  - Total columns: {final_cols}")
print(f"  - Table name: {gold_table_name}")

# Show sample
print("\nSample from gold table:")
display(spark.read.table(gold_table_name).limit(5))

Writing 1970 rows to gold table...

✅ Merged 1970 records into workspace.fotmob.player_overview_gold

Final gold table stats:
  - Total rows: 1970
  - Total columns: 11
  - Table name: workspace.fotmob.player_overview_gold

Sample from gold table:


player_id,player_name,birthdate,club_id,club,primary_position,secondary_positions,preferred_foot,country,data_source,last_updated
180455,Kosovare Asllani,1989-07-29,1075419,London City Lionesses,Attacking Midfielder,"List(Central Midfielder, Striker)",right,Sweden,fotmob_only,2026-07-14T20:34:49.444Z
271109,Alexandra Popp,1991-04-06,394121,VfL Wolfsburg,Striker,"List(Central Midfielder, Attacking Midfielder, Right Winger)",left,Germany,fotmob_only,2026-07-14T20:34:49.444Z
440499,Tuva Hansen,1997-08-04,231497,West Ham United,Center Back,List(Right Back),right,Norway,fotmob_only,2026-07-14T20:34:49.444Z
646111,Noëlle Maritz,1995-12-23,231494,Aston Villa,Center Back,"List(Left Wing-Back, Right Midfielder, Left Winger)",right,Switzerland,fotmob_only,2026-07-14T20:34:49.444Z
734720,Ewa Pajor,1996-12-03,401657,Barcelona,Striker,List(Right Winger),right,Poland,fotmob_only,2026-07-14T20:34:49.444Z
